# 07 - YOLO11s (modern arch + capacity) - chasing > 0.7678

**Objective.** Beat the current project best (**Improved opt = mAP@0.5 0.7678** on the
TEST set; stock-baseline opt 0.7630) with a *different architecture*: **YOLO11s**,
Ultralytics' newer detector at the **small** scale (~9.4M params vs the existing
nano models' ~2-4M).

> ### Iteration 2 - imgsz 800 -> 960
> **v1 (imgsz=800) hit 0.7546 on TEST** - *under* the 0.7678 best. Diagnosis: the bigger
> model overfit the 1440-image train set and the faint low-contrast class **crazing
> collapsed to 0.478** (worse than the nano models), even though strict-IoU localization
> (mAP@.5:.95 = 0.3905) was the best of any model. **This iteration raises imgsz 800 -> 960**
> so the faint crazing / rolled-in_scale textures get more pixels to resolve.
> v1 is preserved at `results/yolo11s_opt/`; this run writes to **`results/yolo11s_960/`**.

**Hypothesis.** A newer architecture + ~3x capacity (both COCO-pretrained) plus higher input
resolution should clear 0.7678, with the gain concentrated on the weak low-contrast classes
(crazing, rolled-in_scale).

**Fairness (apples-to-apples).** Same dataset split (paper 8:1:1 -> 1440/180/180), same
optimized recipe as `updated_03`/`updated_05` (SGD+cosine, 200ep, close_mosaic=20, mixup=0.1,
workers=0, seed=42), same TEST-set eval (plain vs TTA+NMS0.6). The levers vs the other opt
runs are **architecture (YOLO11s)** and **imgsz (960)** - both noted below.

| Technique | Same as opt baseline? | Note |
|---|---|---|
| **imgsz 960** | **no - raised 800 -> 960** | more pixels for faint crazing / rolled-in_scale |
| SGD + cosine, 200ep, patience 60 | yes | identical recipe |
| close_mosaic 20 + mixup 0.1 | yes | identical augmentation stack |
| TTA + NMS 0.6 eval | yes | identical final-squeeze eval |
| **architecture = YOLO11s** | **no - this is the lever** | newer arch + small-scale capacity |

> **One-line architecture swap** (cell "Build model"): set `MODEL = 'yolo11n.pt'`
> (lightweight - less overfit on tiny data), `'yolo11m.pt'`, `'yolo12s.pt'`, `'yolo26n.pt'`,
> or `'yolov8s.pt'` to compare other architectures on the *identical* recipe.

## 1. Check environment
> **Kernel:** the venv with Ultralytics - `C:/Users/student/Downloads/files/.venv`
> (`.venv (Python 3.10.8)`).


In [1]:
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
DEVICE = 0 if torch.cuda.is_available() else 'cpu'


PyTorch: 2.6.0+cu124 | CUDA: True
GPU: NVIDIA RTX 2000 Ada Generation


## 2. Locate dataset
YOLO11s is a **stock** Ultralytics architecture - no custom modules and **no
`register()`** needed (unlike the paper model in nb 05 or the LZY model in nb 06).


In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

DATA_CFG = ROOT / 'data' / 'neu-det-yolo' / 'data.yaml'
assert DATA_CFG.exists(), 'Run 01_data_preparation.ipynb first!'
CLASSES = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
print('Data config:', DATA_CFG)


Data config: c:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\data.yaml


## 3. Build the model & load COCO weights
`YOLO('yolo11s.pt')` loads the YOLO11s **architecture and** its COCO-pretrained weights in
one step (the standard transfer-learning entry for stock models). The detection head is
re-initialised for our 6 classes automatically at train time. First run downloads ~19 MB
from Ultralytics (internet needed once); cached thereafter. If a local copy sits in the
project root it is preferred (offline-safe).


In [3]:
from ultralytics import YOLO

MODEL = 'yolo11s.pt'          # <- one-line swap: yolo11n/m, yolo12s, yolo26n, yolov8s ...

local = ROOT / MODEL          # prefer a local copy if present (offline), else auto-download
model = YOLO(str(local) if local.exists() else MODEL)
model.info()                  # YOLO11s ~9.4M params / ~21.5 GFLOPs


YOLO11s summary: 181 layers, 9,458,752 parameters, 0 gradients, 21.7 GFLOPs


(181, 9458752, 0, 21.718374400000002)

## 4. Train (optimized recipe, imgsz=960)
Same recipe as the other opt runs - the levers are the **architecture** and now **imgsz=960**.
`workers=0` is Windows-safe (avoids the `close_mosaic` data-loader deadlock).
> **VRAM:** at **imgsz=960** use **`batch=4`** (your 800px run used batch=8; 4@960 is actually
> *lighter* than 8@800, so it fits ~16 GB comfortably - raise to 6 if you see headroom and no
> OOM). Expect ~1.5x longer than the 800px run (~5-6 h for 200 ep; `patience=60` may stop sooner).

In [4]:
IMGSZ = 960             # raised 800 -> 960: more pixels for faint crazing / rolled-in_scale
BATCH = 4 if DEVICE == 0 else 2   # 960px on 16 GB (800px used 8); raise to 6 if no OOM

results = model.train(
    data=str(DATA_CFG),
    epochs=200,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    cache=True,
    workers=0,            # Windows-safe (avoids close_mosaic deadlock)
    project=str(ROOT / 'results'),
    name='yolo11s_960',   # NEW dir - preserves the 800px run at results/yolo11s_opt
    exist_ok=True,
    optimizer='SGD',      # SGD+momentum (lr0=0.01) - matches the other opt runs
    cos_lr=True,          # cosine LR decay
    patience=60,
    close_mosaic=20,      # last 20 epochs on clean (non-mosaic) images
    mixup=0.1,            # mild regularization for the small dataset
    seed=42,
    plots=True,
)
print('Training done. Best weights:', ROOT / 'results/yolo11s_960/weights/best.pt')

New https://pypi.org/project/ultralytics/8.4.64 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=c:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11s

## 5. Evaluate - ablation: plain vs optimized (TTA + NMS 0.6) on the TEST set
Both reload `best.pt`, so this runs without re-training. Targets to compare against:
**project best (improved_opt) 0.7678**, stock-baseline opt 0.7630, canonical baseline 0.702.


In [7]:
best = ROOT / 'results' / 'yolo11s_960' / 'weights' / 'best.pt'

plain = YOLO(str(best)).val(data=str(DATA_CFG), split='test', verbose=False)
opt   = YOLO(str(best)).val(data=str(DATA_CFG), split='test', augment=True, iou=0.6, verbose=False)

print(f"{'eval':<22}{'mAP@0.5':>10}{'mAP@.5:.95':>12}{'P':>8}{'R':>8}")
print(f"{'plain':<22}{plain.box.map50:>10.4f}{plain.box.map:>12.4f}{plain.box.mp:>8.3f}{plain.box.mr:>8.3f}")
print(f"{'optimized (TTA+NMS.6)':<22}{opt.box.map50:>10.4f}{opt.box.map:>12.4f}{opt.box.mp:>8.3f}{opt.box.mr:>8.3f}")
best50 = max(float(plain.box.map50), float(opt.box.map50))
print(f"\nproject best (improved_opt): 0.7678   |   yolo11s@800: 0.7546   |   this @960: {best50:.4f}   |   delta vs best: {best50 - 0.7678:+.4f}")

Ultralytics 8.4.51  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 143.454.7 MB/s, size: 14.4 KB)
val: Scanning C:\Users\student\Desktop\SteelDefectDetection\data\neu-det-yolo\labels\test.cache... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 3.0it/s 3.9s0.3s
                   all        180        413      0.681       0.69      0.745      0.378
Speed: 3.2ms preprocess, 14.3ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to C:\Users\student\Desktop\SteelDefectDetection\notebooks\runs\detect\val-34
Ultralytics 8.4.51  Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradient

## 6. Per-class table (best eval) & save summary
Writes `results/yolo11s_960/metrics_summary.txt` in the **same format** as the other runs,
so it drops straight into the leaderboard. Watch the **crazing** row - that's the class the
960px bump is meant to recover (it was 0.478 at 800px).

In [8]:
import pandas as pd
best_res = opt if float(opt.box.map50) >= float(plain.box.map50) else plain
tag = 'optimized (TTA+NMS0.6)' if best_res is opt else 'plain'

rows = [{'class': n, 'mAP@0.5': round(float(best_res.box.ap50[i]), 4),
         'mAP@0.5:0.95': round(float(best_res.box.ap[i]), 4)} for i, n in enumerate(CLASSES)]
df = pd.DataFrame(rows)
df.loc[len(df)] = ['ALL (mean)', round(float(best_res.box.map50), 4), round(float(best_res.box.map), 4)]

run_dir = ROOT / 'results' / 'yolo11s_960'; run_dir.mkdir(parents=True, exist_ok=True)
summary = run_dir / 'metrics_summary.txt'
with open(summary, 'w') as f:
    f.write('YOLO11s (imgsz=960, SGD, 200ep, TTA+NMS) - TEST set\n')
    f.write('=' * 70 + '\n')
    f.write(f'best eval    : {tag}\n')
    f.write(f'mAP@0.5      : {best_res.box.map50:.4f}  (yolo11s@800 0.7546; project best improved_opt 0.7678; baseline 0.702)\n')
    f.write(f'mAP@0.5:0.95 : {best_res.box.map:.4f}\n')
    f.write(f'precision    : {best_res.box.mp:.4f}\n')
    f.write(f'recall       : {best_res.box.mr:.4f}\n\nPer-class mAP@0.5:\n')
    for i, n in enumerate(CLASSES):
        f.write(f'  {n:<18}{float(best_res.box.ap50[i]):.4f}\n')
print('saved', summary)
df

saved c:\Users\student\Desktop\SteelDefectDetection\results\yolo11s_960\metrics_summary.txt


,class,mAP@0.5,mAP@0.5:0.95
0,crazing,0.5014,0.2087
1,inclusion,0.8106,0.4754
2,patches,0.9376,0.5978
3,pitted_surface,0.8364,0.4205
4,rolled-in_scale,0.5466,0.2367
5,scratches,0.8418,0.3916
6,ALL (mean),0.7457,0.3885


**YOLO11s @ 960 done.** Weights -> `results/yolo11s_960/weights/best.pt`, metrics ->
`results/yolo11s_960/metrics_summary.txt`. Compare TEST mAP@0.5 to **yolo11s@800 = 0.7546**
and the project best **0.7678**:
- **If it clears 0.7678** -> new project leader; add the row to the final comparison.
- **If still under** -> next single levers to try (one at a time): `optimizer='AdamW'`
  (`lr0=0.001`), `epochs=300`, or drop to `MODEL='yolo11n.pt'` (less capacity to overfit the
  1440-image set - often the sweet spot for small datasets).

The 800px v1 stays at `results/yolo11s_opt/` for the 800-vs-960 comparison.